# WebBaseLoader: web page to RAG pipeline

This notebook downloads a public web page with `WebBaseLoader`, converts it to documents, builds a local FAISS index, and uses Groq to answer from retrieved page content.

## 1. Imports and project paths

`WebBaseLoader` is useful when RAG knowledge lives on a website instead of local files. It fetches HTML and uses Beautiful Soup to extract readable text into LangChain documents. Only use pages you are allowed to fetch and follow their terms and robots policy.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or its notebook folder.')

PROJECT_ROOT = find_project_root()
os.environ.setdefault('USER_AGENT', 'rag-learning-notebooks/1.0')
# Replace this with a page you are permitted to load.
WEB_URL = 'https://www.rfc-editor.org/rfc/rfc1149.txt'
print(f'Loading: {WEB_URL}')

## 2. Load and inspect the web document

`SoupStrainer` optionally limits extraction to selected HTML elements. This RFC is plain text, so it is set to `None`; for an HTML article, use a strainer such as `SoupStrainer('article')`.

In [ ]:
loader = WebBaseLoader(
    web_paths=(WEB_URL,),
    bs_kwargs={'parse_only': None},
    requests_kwargs={'timeout': 30},
)
documents = loader.load()
if not documents or not documents[0].page_content.strip():
    raise ValueError(f'No readable text was loaded from {WEB_URL}')

print(f'Loaded {len(documents)} web document(s).')
print(documents[0].page_content[:500])
print(documents[0].metadata)

## 3. Chunk, embed, and index page text

Long pages are split into focused chunks before embedding. Sentence-transformers produces semantic vectors locally and FAISS retrieves the chunks closest to the question.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks, embeddings)
print(f'Indexed {len(chunks)} chunks.')

## 4. Retrieve context and generate an answer

The question is answered from retrieved web chunks, not from the LLM's general knowledge. Create a project `.env` file containing `GROQ_API_KEY=your_key` before running this cell.

In [ ]:
load_dotenv(PROJECT_ROOT / '.env')
groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    raise RuntimeError('Set GROQ_API_KEY in .env before running the LLM cell.')

question = 'What is the main idea of this document?'
retrieved_docs = vector_store.similarity_search(question, k=3)
context = '\n\n'.join(doc.page_content for doc in retrieved_docs)

prompt = PromptTemplate.from_template(
    'Answer only from the supplied context. If the context is insufficient, say so.\n\n'
    'Context:\n{context}\n\nQuestion: {question}\nAnswer:'
)
llm = ChatGroq(api_key=groq_api_key, model='openai/gpt-oss-20b', temperature=0.1, max_tokens=512)
answer = llm.invoke(prompt.format(context=context, question=question))

print(answer.content)
print('\nRetrieved web sources:', [doc.metadata.get('source') for doc in retrieved_docs])